# 🔥 L1 & L2 Regularization — A/B Test (PyTorch)

**What you will learn:**
- How overfitting looks in training curves
- L2 = weight_decay in PyTorch optimizer (built-in!)
- L1 = manual penalty added to loss (not built-in to optimizers)
- Side-by-side: No Reg | L2 | L1 | L1+L2 (ElasticNet)
- Decision boundaries, weight distributions, lambda sweep

**PyTorch key difference vs TensorFlow:**
```
TF/Keras  → add regularizer= argument to each layer
PyTorch   → L2: pass weight_decay to optimizer
           → L1: manually compute penalty and add to loss
```


In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — Imports
# ─────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {DEVICE}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — Dataset
# ─────────────────────────────────────────────
N_SAMPLES = 200
NOISE     = 0.25

X, y = make_moons(n_samples=N_SAMPLES, noise=NOISE, random_state=42)
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Convert to PyTorch tensors
def to_tensors(X, y):
    return (torch.tensor(X,  dtype=torch.float32).to(DEVICE),
            torch.tensor(y,  dtype=torch.float32).to(DEVICE))

Xtr_t, ytr_t   = to_tensors(X_tr,  y_tr)
Xval_t, yval_t = to_tensors(X_val, y_val)

# DataLoader for mini-batch training
train_ds     = TensorDataset(Xtr_t, ytr_t)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

print(f'Train : {Xtr_t.shape}  |  Val : {Xval_t.shape}')

# Quick plot
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(*X_tr[y_tr==0].T,  c='#4C9BE8', s=25, label='Class 0 (train)', alpha=0.7)
ax.scatter(*X_tr[y_tr==1].T,  c='#E84C4C', s=25, label='Class 1 (train)', alpha=0.7)
ax.scatter(*X_val[y_val==0].T,c='#4C9BE8', s=25, marker='^', label='Class 0 (val)',  alpha=0.4)
ax.scatter(*X_val[y_val==1].T,c='#E84C4C', s=25, marker='^', label='Class 1 (val)',  alpha=0.4)
ax.set_title('Two-Moons Dataset (small + noisy)', fontsize=13, fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Model definition (nn.Module)
# ─────────────────────────────────────────────
class MLP(nn.Module):
    """Over-parameterised MLP — 4 layers × 128 units.
    Capacity >> 140 training samples → easy to overfit.
    """
    def __init__(self):
        super().__init__()
        # nn.Sequential stacks layers; no regularizer argument in PyTorch layers
        # → regularization is handled OUTSIDE the model (in loss or optimizer)
        self.net = nn.Sequential(
            nn.Linear(2, 128),  nn.ReLU(),
            nn.Linear(128, 128),nn.ReLU(),
            nn.Linear(128, 128),nn.ReLU(),
            nn.Linear(128, 128),nn.ReLU(),
            nn.Linear(128, 1),  nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)   # shape: (batch,)


# ── L1 penalty helper ──────────────────────────────────────────────────────
def l1_penalty(model, lam):
    """
    In PyTorch, L1 is NOT built into the optimizer.
    We compute it manually and ADD it to the task loss.
    Formula: lam * sum(|w|) over all weight matrices.
    """
    penalty = torch.tensor(0., device=DEVICE)
    for name, param in model.named_parameters():
        if 'weight' in name:             # kernels only, skip biases
            penalty = penalty + param.abs().sum()
    return lam * penalty


# Quick sanity check
m = MLP().to(DEVICE)
total_params = sum(p.numel() for p in m.parameters())
print(f'Parameters: {total_params:,}  (vs {len(X_tr)} training samples — heavily overparameterised!)')
print(m)

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — Generic training function
# Shows the KEY PyTorch pattern for each regularization type
# ─────────────────────────────────────────────
def train_model(reg_type='none', lam=0.001, epochs=300, patience=30):
    """
    reg_type : 'none' | 'l2' | 'l1' | 'l1_l2'

    PyTorch regularization patterns:
      L2  → weight_decay parameter in Adam  (built-in)
      L1  → manual penalty added to loss    (manual)
      Both→ combine both approaches         (combined)
    """
    model    = MLP().to(DEVICE)
    criterion= nn.BCELoss()          # binary cross-entropy

    # ── KEY DIFFERENCE: L2 goes into the optimizer ──
    weight_decay = lam if reg_type in ('l2', 'l1_l2') else 0.0
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    # weight_decay implements L2 penalty: loss += lambda * ||W||²
    # This is why L2 is also called "weight decay" — weights decay toward 0 each step

    # L1 flag — will be added to loss inside the loop
    use_l1 = reg_type in ('l1', 'l1_l2')
    l1_lam = lam if use_l1 else 0.0

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_loss, best_state, wait = float('inf'), None, 0

    for epoch in range(epochs):
        # ── TRAINING ──
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()           # 1. clear accumulated gradients
            pred = model(xb)                # 2. forward pass
            loss = criterion(pred, yb)      # 3. task loss (BCE)

            # ── L1 penalty added manually to the loss ──
            if use_l1:
                loss = loss + l1_penalty(model, l1_lam)

            loss.backward()                 # 4. backprop
            optimizer.step()                # 5. weight update (L2 applied here via weight_decay)
            epoch_loss += loss.item() * len(xb)

        # ── VALIDATION ──
        model.eval()
        with torch.no_grad():
            vp   = model(Xval_t)
            vloss= criterion(vp, yval_t).item()
            vacc = ((vp > 0.5).float() == yval_t).float().mean().item()
            tp   = model(Xtr_t)
            tacc = ((tp > 0.5).float() == ytr_t).float().mean().item()

        history['train_loss'].append(epoch_loss / len(Xtr_t))
        history['val_loss'].append(vloss)
        history['train_acc'].append(tacc)
        history['val_acc'].append(vacc)

        # Early stopping
        if vloss < best_val_loss:
            best_val_loss = vloss
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
            wait          = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    return model, history

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — Run all 4 A/B variants
# ─────────────────────────────────────────────
LAMBDA   = 0.001
VARIANTS = [
    ('none',  'No Regularization',    '#E84C4C'),
    ('l2',    'L2  (weight_decay)',   '#4C9BE8'),
    ('l1',    'L1  (manual penalty)', '#2ECC71'),
    ('l1_l2', 'L1+L2 ElasticNet',    '#F39C12'),
]

histories = {}
models    = {}

for reg_type, label, color in VARIANTS:
    print(f'Training: {label} ...', end=' ', flush=True)
    m, h = train_model(reg_type=reg_type, lam=LAMBDA)
    histories[reg_type] = h
    models[reg_type]    = m
    best_val = max(h['val_acc'])
    print(f'best val acc = {best_val:.4f}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — Plot A: Loss curves
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
fig.suptitle('Train vs Validation Loss  (gap = overfitting)', fontsize=13, fontweight='bold')

for ax, (reg_type, label, color) in zip(axes, VARIANTS):
    h  = histories[reg_type]
    ep = range(1, len(h['train_loss'])+1)
    ax.plot(ep, h['train_loss'], color=color, lw=2,    label='Train')
    ax.plot(ep, h['val_loss'],   color=color, lw=2, ls='--', label='Val', alpha=0.8)
    ax.fill_between(ep, h['train_loss'], h['val_loss'], alpha=0.15, color=color)
    ax.set_title(label, fontsize=10, fontweight='bold', color=color)
    ax.set_xlabel('Epoch'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

axes[0].set_ylabel('Loss')
plt.tight_layout(); plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — Plot B: Decision boundaries
# ─────────────────────────────────────────────
@torch.no_grad()
def predict_np(model, X_np):
    t = torch.tensor(X_np, dtype=torch.float32).to(DEVICE)
    return model(t).cpu().numpy()

def plot_boundary(ax, model, X, y, title, color):
    h = 0.03
    x0, x1 = X[:,0].min()-.5, X[:,0].max()+.5
    y0, y1 = X[:,1].min()-.5, X[:,1].max()+.5
    xx, yy = np.meshgrid(np.arange(x0,x1,h), np.arange(y0,y1,h))
    Z = predict_np(model, np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu', levels=20)
    ax.contour( xx, yy, Z, levels=[0.5], colors=[color], linewidths=2)
    ax.scatter(*X[y==0].T, c='#4C9BE8', s=20, edgecolors='k', lw=0.3)
    ax.scatter(*X[y==1].T, c='#E84C4C', s=20, edgecolors='k', lw=0.3)
    va = ((predict_np(model, X_val) > 0.5).astype(int) == y_val).mean()
    ax.set_title(f'{title}\nVal acc: {va:.3f}', fontsize=10, fontweight='bold', color=color)
    ax.set_xticks([]); ax.set_yticks([])

X_all = np.vstack([X_tr, X_val])
y_all = np.concatenate([y_tr, y_val])

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Decision Boundaries', fontsize=13, fontweight='bold')
for ax, (reg_type, label, color) in zip(axes, VARIANTS):
    plot_boundary(ax, models[reg_type], X_all, y_all, label, color)
plt.tight_layout(); plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 8 — Plot C: Weight distributions
# L1 → exact zeros (sparsity), L2 → small non-zeros
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Weight Distributions  (L1 → sparsity | L2 → small dense weights)',
             fontsize=12, fontweight='bold')

for ax, (reg_type, label, color) in zip(axes, VARIANTS):
    ws = []
    for name, p in models[reg_type].named_parameters():
        if 'weight' in name:
            ws.append(p.detach().cpu().numpy().ravel())
    all_w = np.concatenate(ws)

    ax.hist(all_w, bins=80, color=color, alpha=0.8, density=True)
    ax.axvline(0, color='black', lw=1.5, ls='--')
    pct = np.mean(np.abs(all_w) < 0.01) * 100
    ax.set_title(f'{label}\n{pct:.1f}% weights ≈ 0', fontsize=10, fontweight='bold', color=color)
    ax.set_xlabel('Weight value'); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 9 — Lambda sweep
# ─────────────────────────────────────────────
lambdas = [0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1]
res = {'l1': [], 'l2': []}

for lam in lambdas:
    for rt in ['l1', 'l2']:
        _, h = train_model(reg_type=rt, lam=lam, epochs=200, patience=20)
        res[rt].append(max(h['val_acc']))
    print(f'λ={lam:.4f} done')

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(lambdas, res['l2'], 'o-', color='#4C9BE8', lw=2, ms=7, label='L2')
ax.semilogx(lambdas, res['l1'], 's-', color='#2ECC71', lw=2, ms=7, label='L1')
ax.set_xlabel('Lambda (log scale)'); ax.set_ylabel('Val Accuracy')
ax.set_title('Lambda Sweep — Bias-Variance Tradeoff', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 10 — Final summary + formula recap
# ─────────────────────────────────────────────
print('=' * 65)
print(f'{"Variant":<22} {"Train Acc":>10} {"Val Acc":>10} {"Gap":>8}')
print('=' * 65)
for reg_type, label, _ in VARIANTS:
    h     = histories[reg_type]
    tr    = max(h['train_acc'])
    va    = max(h['val_acc'])
    gap   = tr - va
    flag  = ' ⚠️  OVERFIT' if gap > 0.08 else ' ✅'
    print(f'{label:<22} {tr:>10.4f} {va:>10.4f} {gap:>8.4f}{flag}')
print('=' * 65)

print("""
PYTORCH PATTERNS RECAP
──────────────────────────────────────────────────────────
L2  → optimizer = Adam(model.parameters(), weight_decay=λ)
      (weight_decay IS L2 — applied automatically in step)

L1  → loss = BCE(pred, y) + λ * Σ |w|    (manual)
      add l1_penalty(model, λ) to your loss before .backward()

L1+L2 → combine both: weight_decay=λ  +  l1_penalty(model, λ)

INTUITION:
  L2 penalty: gradient = 2λW  → weights shrink proportionally
              → small but NON-ZERO weights (dense solution)
  L1 penalty: gradient = λ·sign(W) → constant push toward 0
              → many EXACT ZEROS (sparse solution = feature selection)
""")